# GRPO 9M Direct Fine-Tune (Colab)

Run GRPO fine-tuning from the DeepMind 9M checkpoint using the direct 9M policy path (`use_9m_direct: true`).

This notebook performs:
1. Colab + repo setup
2. Dependency installation
3. Required W&B authentication
4. Download of DeepMind `9M.zip` checkpoint payload
5. JAX -> PyTorch conversion (`jax_9m_converted.pt`)
6. Runtime GRPO config materialization
7. GRPO training launch + artifact verification

Use a GPU runtime in Colab.

In [ ]:
#@title Runtime Parameters
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ARTIFACT_ROOT = "/content/drive/MyDrive/data/grpo-chess/grpo_9m_direct"  #@param {type:"string"}
WANDB_API_KEY = ""  #@param {type:"string"}
RUN_NAME_SUFFIX = ""  #@param {type:"string"}
GRPO_CONFIG_BASE = "grpo_9m_direct.yaml"  #@param {type:"string"}
RUN_FULL_DEFAULT = True  #@param {type:"boolean"}
NUM_EPOCHS_OVERRIDE = 0  #@param {type:"integer"}
BATCH_SIZE_OVERRIDE = 0  #@param {type:"integer"}
STEPS_PER_EPOCH_OVERRIDE = 0  #@param {type:"integer"}
NUM_TRAJ_OVERRIDE = 0  #@param {type:"integer"}
TRAJ_DEPTH_OVERRIDE = 0  #@param {type:"integer"}
EVAL_GAMES_OVERRIDE = 0  #@param {type:"integer"}

In [ ]:
# Setup workspace and clone repo
import os
import shutil
import subprocess
import sys
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

repo = Path('/content/grpo_chess')
if repo.exists():
    shutil.rmtree(repo)

subprocess.check_call(['git', 'clone', REPO_URL, str(repo)])
os.chdir(repo)
subprocess.check_call(['git', 'fetch', '--all', '--tags'])
subprocess.check_call(['git', 'checkout', REPO_REF])
subprocess.check_call(['git', 'submodule', 'update', '--init', '--recursive'])

if str(repo) not in sys.path:
    sys.path.append(str(repo))

print('Repo ready at', repo)
print('Python:', sys.version.split()[0])

In [ ]:
# Install dependencies (runtime restarts once after first install)
import os
import signal
from pathlib import Path

deps_ready = Path('/tmp/grpo_9m_colab_deps_ready')
if not deps_ready.exists():
    %pip install -q --upgrade pip setuptools wheel
    !grep -vE '^numpy==' requirements.txt > /tmp/requirements-colab.txt
    %pip install -q -r /tmp/requirements-colab.txt
    %pip install -q --force-reinstall --no-cache-dir \
        "numpy==2.1.3" "scipy==1.14.1" "pandas==2.2.2" "pyarrow==18.1.0" \
        "requests==2.32.4" "urllib3<=2.5.0" "jedi>=0.19.1"
    %pip install -q "jax==0.4.33" "jaxlib==0.4.33" "dm-haiku" "chex" "optax" "orbax-checkpoint"
    !apt-get -qq update
    !apt-get -qq install -y stockfish
    !which stockfish
    deps_ready.write_text('ok')
    print('Dependencies installed. Restarting runtime to ensure a clean binary state...')
    os.kill(os.getpid(), signal.SIGKILL)
else:
    !which stockfish
    import numpy, scipy, pandas, pyarrow, jax, haiku, orbax.checkpoint
    print('numpy', numpy.__version__)
    print('scipy', scipy.__version__)
    print('pandas', pandas.__version__)
    print('pyarrow', pyarrow.__version__)
    print('jax', jax.__version__)
    print('haiku', haiku.__version__)
    print('orbax.checkpoint', orbax.checkpoint.__version__)

In [ ]:
# Required W&B auth
import os

wandb_key = WANDB_API_KEY.strip()
if not wandb_key:
    try:
        from google.colab import userdata
        wandb_key = (userdata.get('WANDB_API_KEY') or userdata.get('WANDB_KEY') or '').strip()
    except Exception:
        wandb_key = ''

if not wandb_key:
    raise RuntimeError('W&B key is required. Set WANDB_API_KEY runtime param or Colab Secret WANDB_API_KEY/WANDB_KEY.')

os.environ['WANDB_API_KEY'] = wandb_key
os.environ['WANDB_KEY'] = wandb_key
print('W&B key configured.')

In [ ]:
# Artifact paths + download DeepMind 9M checkpoint payload
from pathlib import Path
import urllib.request
import zipfile

artifact_root = Path(DRIVE_ARTIFACT_ROOT if USE_DRIVE else '/content/artifacts/grpo_9m_direct')
searchless_ckpt_root = artifact_root / 'searchless_ckpts'
converted_ckpt_root = artifact_root / 'converted_ckpts'
grpo_runs_root = artifact_root / 'grpo_runs'

for d in [artifact_root, searchless_ckpt_root, converted_ckpt_root, grpo_runs_root]:
    d.mkdir(parents=True, exist_ok=True)

nine_m_dir = searchless_ckpt_root / '9M'
zip_path = searchless_ckpt_root / '9M.zip'
download_url = 'https://storage.googleapis.com/searchless_chess/checkpoints/9M.zip'

if not nine_m_dir.exists():
    print('Downloading', download_url)
    urllib.request.urlretrieve(download_url, zip_path)
    print('Extracting', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(searchless_ckpt_root)
    if zip_path.exists():
        zip_path.unlink()
else:
    print('Reusing existing 9M checkpoint directory:', nine_m_dir)

expected_orbax = nine_m_dir / '6400000' / 'params' / 'checkpoint'
if not expected_orbax.exists():
    raise FileNotFoundError(f'Missing expected Orbax checkpoint file: {expected_orbax}')

print('Artifact root:', artifact_root)
print('Expected Orbax checkpoint found:', expected_orbax)

In [ ]:
# Convert JAX 9M checkpoint -> PyTorch state dict
import subprocess
import sys

converted_ckpt_path = converted_ckpt_root / 'jax_9m_converted.pt'
if not converted_ckpt_path.exists():
    cmd = [
        sys.executable, '-m', 'src.distill.convert_jax_checkpoint',
        '--checkpoint_dir', str(searchless_ckpt_root),
        '--model_name', '9M',
        '--step', '6400000',
        '--output', str(converted_ckpt_path),
    ]
    print('Running conversion command:')
    print(' '.join(cmd))
    subprocess.check_call(cmd)
else:
    print('Reusing converted checkpoint:', converted_ckpt_path)

if not converted_ckpt_path.exists():
    raise FileNotFoundError(f'Conversion did not produce checkpoint: {converted_ckpt_path}')

print('Converted checkpoint:', converted_ckpt_path)
print('Size (MB):', round(converted_ckpt_path.stat().st_size / (1024 * 1024), 2))

In [ ]:
# Build runtime GRPO config from src/configs/grpo_9m_direct.yaml
from datetime import datetime
import re
import yaml

base_config_path = repo / 'src' / 'configs' / GRPO_CONFIG_BASE
if not base_config_path.exists():
    raise FileNotFoundError(f'Base GRPO config not found: {base_config_path}')

with base_config_path.open('r') as f:
    cfg = yaml.safe_load(f)

if not RUN_FULL_DEFAULT:
    cfg['training']['num_epochs'] = 1
    cfg['training']['batch_size'] = 4
    cfg['training']['steps_per_epoch'] = 16
    cfg['grpo']['num_trajectories'] = 2
    cfg['grpo']['trajectory_depth'] = 4
    cfg['eval']['games'] = 4

def _apply_positive_int_override(raw_value, section, key):
    if isinstance(raw_value, int) and raw_value > 0:
        section[key] = int(raw_value)

_apply_positive_int_override(NUM_EPOCHS_OVERRIDE, cfg['training'], 'num_epochs')
_apply_positive_int_override(BATCH_SIZE_OVERRIDE, cfg['training'], 'batch_size')
_apply_positive_int_override(STEPS_PER_EPOCH_OVERRIDE, cfg['training'], 'steps_per_epoch')
_apply_positive_int_override(NUM_TRAJ_OVERRIDE, cfg['grpo'], 'num_trajectories')
_apply_positive_int_override(TRAJ_DEPTH_OVERRIDE, cfg['grpo'], 'trajectory_depth')
_apply_positive_int_override(EVAL_GAMES_OVERRIDE, cfg['eval'], 'games')

stockfish_path = '/usr/games/stockfish'
if not Path(stockfish_path).exists():
    raise FileNotFoundError(f'Stockfish binary missing at {stockfish_path}')
cfg['stockfish']['path'] = stockfish_path
cfg['pretrain']['checkpoint_path'] = str(converted_ckpt_path)
cfg['pretrain']['use_9m_direct'] = True

timestamp = datetime.utcnow().strftime('%Y%m%d-%H%M%S')
suffix = RUN_NAME_SUFFIX.strip()
if suffix:
    safe_suffix = re.sub(r'[^a-zA-Z0-9_.-]+', '-', suffix)
    run_name = f'grpo-9m-direct-{timestamp}-{safe_suffix}'
else:
    run_name = f'grpo-9m-direct-{timestamp}'

run_checkpoint_dir = grpo_runs_root / run_name
run_checkpoint_dir.mkdir(parents=True, exist_ok=True)
cfg['training']['checkpoint_dir'] = str(run_checkpoint_dir)
cfg['training']['use_wandb'] = True
cfg['training']['wandb_project'] = 'Chess-GRPO-Bot'

runtime_config_name = 'grpo_9m_colab_runtime.yaml'
runtime_config_path = repo / 'src' / 'configs' / runtime_config_name
with runtime_config_path.open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Runtime config saved:', runtime_config_path)
print('checkpoint_dir:', cfg['training']['checkpoint_dir'])
print('num_epochs:', cfg['training']['num_epochs'])
print('batch_size:', cfg['training']['batch_size'])
print('steps_per_epoch:', cfg['training']['steps_per_epoch'])
print('num_trajectories:', cfg['grpo']['num_trajectories'])
print('trajectory_depth:', cfg['grpo']['trajectory_depth'])
print('eval.games:', cfg['eval']['games'])
print('pretrain.checkpoint_path:', cfg['pretrain']['checkpoint_path'])

In [ ]:
# Launch GRPO training
import torch
from src.train_self_play import train as grpo_train

torch.set_float32_matmul_precision('high')
print('Launching GRPO with config:', runtime_config_name)
grpo_train(
    config_path=runtime_config_name,
    dataloader_kwargs={'num_workers': 0},
)

In [ ]:
# Verify artifacts and surface run pointers
import json

ckpt_files = sorted(run_checkpoint_dir.glob('*.ckpt'), key=lambda p: p.stat().st_mtime)
if not ckpt_files:
    raise RuntimeError(f'No .ckpt files found in {run_checkpoint_dir}')

latest_ckpt = ckpt_files[-1]
print('Checkpoint dir:', run_checkpoint_dir)
print('Num checkpoints:', len(ckpt_files))
print('Latest checkpoint:', latest_ckpt)

run_url = os.environ.get('WANDB_RUN_URL', '').strip()
if run_url:
    print('W&B run URL:', run_url)
else:
    metadata_files = sorted((repo / 'wandb').glob('**/wandb-metadata.json'), key=lambda p: p.stat().st_mtime)
    if metadata_files:
        latest_meta = metadata_files[-1]
        try:
            payload = json.loads(latest_meta.read_text())
            guessed_url = payload.get('url') or payload.get('run_url')
            if guessed_url:
                print('W&B run URL (metadata):', guessed_url)
            else:
                print('Latest W&B metadata file:', latest_meta)
        except Exception:
            print('Latest W&B metadata file:', latest_meta)
    else:
        print('No local W&B metadata found under', repo / 'wandb')

print('Run complete.')

## Troubleshooting

- **W&B key missing**: set `WANDB_API_KEY` in runtime params or add Colab Secret `WANDB_API_KEY`/`WANDB_KEY`.
- **JAX conversion import errors**: rerun dependency cell and ensure runtime restarted after install.
- **Stockfish path errors**: verify `/usr/games/stockfish` exists; rerun apt install cell if needed.
- **OOM / slow training**: reduce `BATCH_SIZE_OVERRIDE`, `NUM_TRAJ_OVERRIDE`, `TRAJ_DEPTH_OVERRIDE`, and `STEPS_PER_EPOCH_OVERRIDE`.
- **Fresh smoke pass**: set `NUM_EPOCHS_OVERRIDE=1`, `STEPS_PER_EPOCH_OVERRIDE=8`, `NUM_TRAJ_OVERRIDE=2`, `TRAJ_DEPTH_OVERRIDE=4`, `EVAL_GAMES_OVERRIDE=2`.